# Spark Bronze wikpedia page reads

In [1]:
exeuction_date = "2025-01-01"
full_refresh = True

In [2]:
import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [3]:

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg


def get_spark_session():
    """Get spark client for s3."""
    s3_cfg = get_config("s3")
    spark_cfg = get_config("spark")

    print(f"using s3 endpoint: {s3_cfg['url']}")
    print(f"using spark master: {spark_cfg['master_url']}")

    spark_session = (  SparkSession
        .builder
        .master('spark://spark-master:7077')
        .appName("Wikipedia page reads - Bronze")
        .config("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .config("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
        .config("spark.hadoop.fs.s3a.endpoint", s3_cfg["url"])
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config(
            "spark.jars",
            "../jars/hadoop-aws-3.3.4.jar,../jars/aws-java-sdk-bundle-1.12.262.jar")
        .getOrCreate()
    )

    return spark_session


In [ ]:
from pyspark.sql.functions import input_file_name, col, sum as _sum

spark = get_spark_session()




s3_path = "s3a://data/test_small"
s3_path = "s3a://data/wikipedia_pageviews/2024/2024-12/31/pageviews-20241231-000000.gz"
s3_path = "s3a://data/test-20241231-000000-mini.txt"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-01/10/*.gz"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/12/pageviews-20250312-000000.gz"

schema = StructType([
    StructField(name="domain_code", dataType=StringType(), nullable = True),
    StructField("page_title", StringType(), True),
    StructField("count_views", StringType(), True),
    StructField("total_response_size", StringType(), True),
])
spark.sparkContext.setLogLevel("INFO")
print(f"Reading data from {s3_path}")

df = (
    spark.read.format("csv")
    .option("delimiter", " ")
    .option("header", "false")
    .option("inferSchema", "false")
    .schema(schema)
    .load(s3_path)
)

df_org = df.withColumn("file_name", input_file_name())


df = df_org

df.show(10)

filtered_df = ( df
               .filter(col("domain_code").startswith("sv"))
)

aggregated_df = (
    filtered_df.groupBy("page_title", "domain_code")
    .agg(
        _sum(col("count_views").cast("int")).alias("total_count_views"),
    )
)

try:
    aggregated_df.orderBy(col("total_count_views").desc()).show()
except Exception as e:
    print(f"Error displaying DataFrame: {e}")


spark.stop()

retrieving s3 config from http://tfds-config:8005/api/configs/s3
retrieving spark config from http://tfds-config:8005/api/configs/spark
using s3 endpoint: http://s3-minio:9000
using spark master: spark://spark-master:7077


25/04/07 20:07:21 WARN Utils: Your hostname, McJens.local resolves to a loopback address: 127.0.0.1; using 192.168.0.152 instead (on interface en0)
25/04/07 20:07:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/04/07 20:07:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Reading data from s3a://data/wikipedia_pageviews/2025/2025-03/12/pageviews-20250312-000000.gz


25/04/07 20:07:24 DEBUG FileSystem: Looking for FS supporting file
25/04/07 20:07:24 DEBUG FileSystem: looking for configuration option fs.file.impl
25/04/07 20:07:24 DEBUG FileSystem: Looking in service filesystems for implementation class
25/04/07 20:07:24 DEBUG FileSystem: FS for file is class org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem
25/04/07 20:07:24 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
25/04/07 20:07:24 DEBUG SharedState: Applying other initial session options to HadoopConf: spark.jars -> ../jars/hadoop-aws-3.3.4.jar,../jars/aws-java-sdk-bundle-1.12.262.jar
25/04/07 20:07:24 DEBUG SharedState: Applying other initial session options to HadoopConf: spark.hadoop.fs.s3a.path.style.access -> true
25/04/07 20:07:24 DEBUG SharedState: Applying other initial session options to HadoopConf: spark.app.name -> Wikipedia page reads - Bronze
25/04/07 20:07:24 DEBUG SharedState: Applying other initial session options to

+-----------+
|domain_code|
+-----------+
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
+-----------+
only showing top 50 rows



25/04/07 20:07:33 DEBUG ResolveReferencesInSort: Resolving 'total_count_views to total_count_views#28L
25/04/07 20:07:33 DEBUG ContextCleaner: Got cleaning task CleanAccum(27)
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaning accumulator 27
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaned accumulator 27
25/04/07 20:07:33 DEBUG ContextCleaner: Got cleaning task CleanAccum(39)
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaning accumulator 39
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaned accumulator 39
25/04/07 20:07:33 DEBUG ContextCleaner: Got cleaning task CleanAccum(45)
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaning accumulator 45
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaned accumulator 45
25/04/07 20:07:33 DEBUG ContextCleaner: Got cleaning task CleanAccum(53)
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaning accumulator 53
25/04/07 20:07:33 DEBUG ContextCleaner: Cleaned accumulator 53
25/04/07 20:07:33 DEBUG ContextCleaner: Got cleaning task CleanAccum(43)
25/04/07 20:07:33 DEBUG 

+--------------------+-----------+-----------------+
|          page_title|domain_code|total_count_views|
+--------------------+-----------+-----------------+
|    Portal:Huvudsida|       sv.m|              554|
|        Jan_Stenbeck|       sv.m|              378|
|    Portal:Huvudsida|         sv|              355|
|         Special:Sök|       sv.d|              274|
|                   -|       sv.d|              226|
|      Carl_Lundström|       sv.m|              199|
|Margaretha_af_Ugglas|       sv.m|              192|
|         Special:Sök|         sv|              185|
|   Cristina_Stenbeck|       sv.m|              152|
|         Special:Sök|       sv.v|              145|
|    Kaj_(humorgrupp)|       sv.m|              103|
|            Kinnevik|       sv.m|               87|
|        Max_Stenbeck|       sv.m|               84|
|         Special:Sök|       sv.m|               82|
|      Jakob_Norrgård|       sv.m|               82|
|         Special:Sök|     sv.m.v|            

25/04/07 20:07:41 INFO MapOutputTrackerMasterEndpoint: MapOutputTrackerMasterEndpoint stopped!
25/04/07 20:07:41 INFO MemoryStore: MemoryStore cleared
25/04/07 20:07:41 INFO BlockManager: BlockManager stopped
25/04/07 20:07:41 INFO BlockManagerMaster: BlockManagerMaster stopped
25/04/07 20:07:41 INFO OutputCommitCoordinator$OutputCommitCoordinatorEndpoint: OutputCommitCoordinator stopped!
25/04/07 20:07:41 DEBUG PoolThreadCache: Freed 3 thread-local buffer(s) from thread: rpc-server-4-1
25/04/07 20:07:41 DEBUG PoolThreadCache: Freed 3 thread-local buffer(s) from thread: rpc-server-4-2
25/04/07 20:07:41 INFO SparkContext: Successfully stopped SparkContext


25/04/07 20:07:43 DEBUG PoolThreadCache: Freed 7 thread-local buffer(s) from thread: rpc-server-4-4
25/04/07 20:07:43 DEBUG PoolThreadCache: Freed 4 thread-local buffer(s) from thread: shuffle-server-7-1
25/04/07 20:07:43 DEBUG PoolThreadCache: Freed 4 thread-local buffer(s) from thread: rpc-server-4-5
25/04/07 20:07:43 DEBUG PoolThreadCache: Freed 9 thread-local buffer(s) from thread: rpc-server-4-3
25/04/07 20:07:43 DEBUG PoolThreadCache: Freed 4 thread-local buffer(s) from thread: rpc-server-4-6
25/04/07 20:08:26 DEBUG PoolingHttpClientConnectionManager: Closing connections idle longer than 60000 MILLISECONDS
